# Data Engineering: Flatfox Listings in Central Switzerland

This notebook turns the raw Flatfox scrapes into a dataset of all listings in Central Switzerland (Lucerne, Uri, Schwyz, Obwalden, Nidwalden, Zug), enriched with municipality indicators.

## Approach

1. **Merge the scrapes:** All `listings_all_fields.csv` files under `data/flatfox_data/` are combined. If a listing appears in several runs, the most recent version is kept. `first_seen_at`, `last_seen_at` and `scrape_count` record when and how often it was seen. The merged result is saved as `flatfox_listings_merged.csv`.
2. **Assign municipality and canton:** Addresses on Flatfox are not always complete, so every listing is located by its coordinates instead. A spatial join with the municipality boundaries of swissBOUNDARIES3D (layer `tlm_hoheitsgebiet`, lake areas and communal lands excluded) gives the BFS municipality number. The municipality name and canton are derived from it. `gemeinde_geo_status` documents whether the assignment succeeded. Afterwards, only listings in the six Central Swiss cantons are kept.
3. **Add municipality indicators:** The indicators from `gemeinde_indikatoren_zentralschweiz.csv` (created by `data_engineering_municipality`) are merged by BFS municipality number. `municipality_match_status` shows whether a match was found.

Apart from the restriction to Central Switzerland, no listings are removed. All offer types and categories (including sales, parking spaces and inactive listings) are kept. All filtering by listing characteristics is done in `data_cleaning_rentals`.

## Output

| File | Content |
|---|---|
| `data/data_processed/flatfox_listings_merged.csv` | All scraped listings, one row per listing ID |
| `data/data_processed/flatfox_rentals_zentralschweiz.csv` | All listings in Central Switzerland with municipality indicators, input for `data_cleaning_rentals` |

Run `data_engineering_municipality` first, because this notebook needs its output.

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [2]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
output_file = DATA_DIR / "data_processed" / "flatfox_rentals_zentralschweiz.csv"
gpkg_path = DATA_DIR / "geodata" / "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg"

# Created by data_engineering_municipality.ipynb.
municipality_file = DATA_DIR / "data_processed" / "gemeinde_indikatoren_zentralschweiz.csv"

CENTRAL_CANTONS = ["LU", "UR", "SZ", "OW", "NW", "ZG"]


In [3]:
def merge_flatfox_csvs(data_dir):
    """Read all scrape CSVs, remove identical duplicates and keep the most recent row per listing ID."""
    
    data_dir = Path(data_dir)
    source_files = sorted((data_dir / "flatfox_data").glob("*/listings_all_fields.csv"))
    output_file = (data_dir / "data_processed" / "flatfox_listings_merged.csv")

    if not source_files:
        raise FileNotFoundError("No scrape files found.")

    frames = []
    report = []

    for source in source_files:
        frame = pd.read_csv(
            source,
            encoding="utf-8-sig",
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )

        frame["pk"] = frame["pk"].str.strip()

        if (frame["pk"].isna() | frame["pk"].eq("")).any():
            raise ValueError(f"Empty listing ID in {source}")

        rows_read = len(frame)
        frame = frame.drop_duplicates()

        if frame["pk"].duplicated().any():
            raise ValueError(
                f"Conflicting rows with the same pk in {source}"
            )

        report.append({
            "source_run": source.parent.name,
            "rows_read": rows_read,
            "unique_ids": len(frame),
            "identical_duplicates": rows_read - len(frame),
        })

        frame["source_run"] = source.parent.name
        frame["source_file"] = source.relative_to(data_dir).as_posix()
        frame["__run_at"] = pd.to_datetime(
            source.parent.name,
            format="%Y%m%dT%H%M%S_%fZ",
            utc=True,
        )
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined = combined.sort_values("__run_at", kind="stable")

    if combined.empty:
        raise ValueError("The scrape files contain no listings.")

    # Determine sightings across all available scrapes.
    grouped = combined.groupby("pk")

    combined["first_seen_at"] = grouped["__run_at"].transform("min")
    combined["last_seen_at"] = grouped["__run_at"].transform("max")
    combined["scrape_count"] = grouped["source_run"].transform("nunique")

    # Keep the entire most recent row per ID.
    merged = (
        combined.drop_duplicates("pk", keep="last")
        .drop(columns="__run_at")
        .sort_values("pk")
        .reset_index(drop=True)
    )

    for column in ["first_seen_at", "last_seen_at"]:
        merged[column] = merged[column].dt.strftime(
            "%Y-%m-%dT%H:%M:%S.%fZ"
        )

    # Write the complete file first, then replace the previous result.
    output_file.parent.mkdir(parents=True, exist_ok=True)
    temporary_file = output_file.with_suffix(".csv.tmp")
    merged.to_csv(temporary_file, index=False, encoding="utf-8-sig")
    temporary_file.replace(output_file)

    report_df = pd.DataFrame(report)

    print(f"Scrape files: {len(source_files)}")
    print(f"Rows read: {report_df['rows_read'].sum()}")
    print(f"Unique listings: {len(merged)}")
    print(f"Saved: {output_file}")

    return merged, report_df

In [4]:
def assign_municipalities(df, gpkg_path):
    """
    Assign each listing to a municipality based on its coordinates, derive
    the canton from it, document missing or ambiguous assignments and
    restrict the data to Central Switzerland.
    """
    gpkg_path = Path(gpkg_path)

    if not gpkg_path.is_file():
        raise FileNotFoundError(f"GeoPackage file not found: {gpkg_path}")

    listings = df.copy().reset_index(drop=True)

    # Swiss municipal territories only, no lake areas or communal lands (Kommunanzen).
    municipalities = gpd.read_file(gpkg_path, layer="tlm_hoheitsgebiet")
    municipalities = municipalities.loc[
        municipalities["icc"].eq("CH")
        & municipalities["objektart"].eq("Gemeindegebiet"),
        ["bfs_nummer", "name", "kantonsnummer", "geometry"],
    ].copy()

    if municipalities.empty:
        raise ValueError("No Swiss municipal territories found in the geodata.")

    canton_codes = dict(enumerate(
        [
            "ZH", "BE", "LU", "UR", "SZ", "OW", "NW", "GL", "ZG",
            "FR", "SO", "BS", "BL", "SH", "AR", "AI", "SG", "GR",
            "AG", "TG", "TI", "VD", "VS", "NE", "GE", "JU",
        ],
        start=1,
    ))

    municipalities["gemeinde_bfs"] = municipalities["bfs_nummer"].astype("Int64")
    municipalities["kanton_code"] = municipalities["kantonsnummer"].map(canton_codes)

    if municipalities["kanton_code"].isna().any():
        raise ValueError("Unknown canton number in the geodata.")

    # Prepare listing coordinates
    longitude = pd.to_numeric(listings["longitude"], errors="coerce")
    latitude = pd.to_numeric(listings["latitude"], errors="coerce")
    valid_coordinates = longitude.between(-180, 180) & latitude.between(-90, 90)

    points = gpd.GeoDataFrame(
        index=listings.index[valid_coordinates],
        geometry=gpd.points_from_xy(
            longitude[valid_coordinates], latitude[valid_coordinates]
        ),
        crs="EPSG:4326",
    ).to_crs(municipalities.crs)

    matches = gpd.sjoin(
        points,
        municipalities[["gemeinde_bfs", "geometry"]],
        how="left",
        predicate="intersects",
    )

    # Several geometries with the same BFS number count as one municipality.
    candidates = (
        matches.groupby(level=0)["gemeinde_bfs"]
        .agg(lambda values: tuple(values.dropna().unique()))
        .reindex(listings.index)
    )

    listings["gemeinde_bfs"] = candidates.map(
        lambda values: values[0] if isinstance(values, tuple) and len(values) == 1 else pd.NA
    ).astype("Int64")

    status = pd.Series("koordinaten_ungueltig", index=listings.index)
    status[valid_coordinates] = "kein_treffer"
    status[candidates.map(lambda values: isinstance(values, tuple) and len(values) > 1)] = "mehrdeutiger_treffer"
    status[listings["gemeinde_bfs"].notna()] = "zugeordnet"
    listings["gemeinde_geo_status"] = status

    # Derive name and canton directly from the assigned municipality
    municipality_info = municipalities.drop_duplicates("gemeinde_bfs").set_index("gemeinde_bfs")
    listings["gemeinde_name"] = listings["gemeinde_bfs"].map(municipality_info["name"]).astype("string")
    listings["kanton_geo"] = listings["gemeinde_bfs"].map(municipality_info["kanton_code"]).astype("string")

    # Restrict to Central Switzerland
    df_central = listings.loc[listings["kanton_geo"].isin(CENTRAL_CANTONS)].copy()

    return listings, df_central

In [5]:
def add_municipality_indicators(df_rentals, municipality_file):
    """Add the municipality indicators (incl. median income) by BFS municipality number and keep all listings."""
    indicators = pd.read_csv(municipality_file, encoding="utf-8-sig")

    # Name and canton already come from the geographic assignment of the listings.
    indicators = indicators.drop(columns=["gemeinde_name", "kanton_geo"])
    indicators["gemeinde_bfs"] = indicators["gemeinde_bfs"].astype("Int64")

    rentals = df_rentals.copy()
    rentals["gemeinde_bfs"] = pd.to_numeric(
        rentals["gemeinde_bfs"], errors="raise"
    ).astype("Int64")

    return rentals.merge(
        indicators,
        on="gemeinde_bfs",
        how="left",
        validate="many_to_one",
        indicator="municipality_match_status",
        sort=False,
    )

In [6]:
# 1. Merge scrape files
df_merged, merge_report = merge_flatfox_csvs(DATA_DIR)

Scrape files: 14
Rows read: 42488
Unique listings: 4292
Saved: /Users/enrue/Developer/jupyter-setup/cip_projekt/data/data_processed/flatfox_listings_merged.csv


In [7]:
# 2. Assign cantons and municipalities
df_geo, df_central = assign_municipalities(df_merged, gpkg_path)

In [8]:
# 3. Add municipality indicators
df_central = add_municipality_indicators(df_central, municipality_file)

output_file.parent.mkdir(parents=True, exist_ok=True)
df_central.to_csv(output_file, index=False, encoding="utf-8-sig")

In [9]:
print(f"\nMerged listings: {len(df_merged)}")
print(f"Thereof in Central Switzerland: {len(df_central)}")
print(f"CSV saved: {output_file}")
print("\nStatus of municipality assignment:")

display(df_geo["gemeinde_geo_status"].value_counts())

print("\nListings per Central Swiss canton:")
display(df_central["kanton_geo"].value_counts().reindex(CENTRAL_CANTONS, fill_value=0))

print("\nStatus of municipality indicator match:")
display(df_central["municipality_match_status"].value_counts())

print("\nMunicipalities without income value:")
display(df_central.loc[df_central["income_median_chf"].isna(), ["gemeinde_bfs", "gemeinde_name"],].drop_duplicates())

display(df_central)


Merged listings: 4292
Thereof in Central Switzerland: 2640
CSV saved: /Users/enrue/Developer/jupyter-setup/cip_projekt/data/data_processed/flatfox_rentals_zentralschweiz.csv

Status of municipality assignment:


gemeinde_geo_status
zugeordnet    4292
Name: count, dtype: int64


Listings per Central Swiss canton:


kanton_geo
LU    1567
UR      80
SZ     355
OW      59
NW      66
ZG     513
Name: count, dtype: Int64


Status of municipality indicator match:


municipality_match_status
both          2640
left_only        0
right_only       0
Name: count, dtype: int64


Municipalities without income value:


,gemeinde_bfs,gemeinde_name


,pk,slug,url,short_url,submit_url,status,created,offer_type,object_category,object_type,...,social_assistance_estimated,social_assistance_rate_pct_year,social_assistance_rate_pct_gemeindestand,income_year,income_metric,employment_per_100_residents_year,employment_source_gemeindestand,employment_population_gemeindestand,employment_boundary_adjustment,municipality_match_status
0,1022104,acherweg-4-hagenstrasse-42a-42b-6460-altdorf,/en/flat/acherweg-4-hagenstrasse-42a-42b-6460-...,/1022104/,/en/listing/1022104/submit/,act,2023-11-07T19:34:16.087594+01:00,RENT,PARK,COVERED_PARKING_PLACE_BIKE,...,False,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
1,1055027,moschtiweg-2-ehp-6212-st-erhard,/en/flat/moschtiweg-2-ehp-6212-st-erhard/1055027/,/1055027/,/en/listing/1055027/submit/,act,2023-12-15T14:12:46.992147+01:00,RENT,PARK,GARAGE_SLOT,...,True,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
2,1060721,hitzkircherstrasse-6-6284-hitzkirch,/en/flat/hitzkircherstrasse-6-6284-hitzkirch/1...,/1060721/,/en/listing/1060721/submit/,act,2023-12-22T11:28:41.800640+01:00,RENT,PARK,GARAGE_SLOT,...,True,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
3,1086630,ettiswilerstr-12-6130-willisau,/en/flat/ettiswilerstr-12-6130-willisau/1086630/,/1086630/,/en/listing/1086630/submit/,act,2024-01-22T17:27:21.095733+01:00,RENT,INDUSTRY,ADVERTISING_AREA,...,True,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
4,1110748,hauptstrasse-50-6436-muotathal,/en/flat/hauptstrasse-50-6436-muotathal/1110748/,/1110748/,/en/listing/1110748/submit/,act,2024-02-16T14:57:10.983543+01:00,RENT,INDUSTRY,COMMERCIAL,...,False,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2635,86519569,wurzenbachmatte-34-6006-luzern,/en/flat/wurzenbachmatte-34-6006-luzern/86519569/,/86519569/,/en/listing/86519569/submit/,act,2026-11-05T21:22:47.337684+01:00,RENT,APARTMENT,APARTMENT,...,True,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
2636,86519678,im-latten-5-6110-wolhusen,/en/flat/im-latten-5-6110-wolhusen/86519678/,/86519678/,/en/listing/86519678/submit/,act,2026-11-05T23:21:32.607549+01:00,RENT,APARTMENT,APARTMENT,...,True,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
2637,86519974,6020-emmenbrucke,/en/flat/6020-emmenbrucke/86519974/,/86519974/,/en/listing/86519974/submit/,act,2026-11-06T06:56:04.302569+01:00,SALE,APARTMENT,APARTMENT,...,True,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
2638,945823,alpenstrasse-15-6300-zug,/en/flat/alpenstrasse-15-6300-zug/945823/,/945823/,/en/listing/945823/submit/,act,2023-08-21T10:05:42.306045+02:00,RENT,PARK,GARAGE_SLOT,...,True,2024,01.01.2024,2022,median_steink,2024,01.01.2024,04.06.2025,No merger aggregation applied,both
